# CSE 432 — Midterm 2 Comprehensive Practice Workbook
## Weeks 6–10: Linear Regression → Model Validation

**Instructions:**
- This workbook covers **every major topic** from Weeks 6–10.
- Each section has **conceptual questions** (answer in markdown) AND **implementation questions** (write + run code).
- Treat this like a guided study session — work through each part, check your understanding, then move on.
- Use `SEED` for all `random_state` parameters.

**Format for each question:**
1. A code cell marked `# YOUR CODE HERE` — write and execute your code
2. A markdown cell marked `YOUR ANSWER HERE` — explain your results in words

**Tip:** If you get stuck on a question, write what you *think* the answer is and flag your uncertainty. Partial credit > blank.

---

## Setup — DO NOT EDIT

In [1]:
# DO NOT EDIT THIS CELL
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, cross_validate,
                                      StratifiedKFold, KFold, GridSearchCV,
                                      validation_curve)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import (LinearRegression, Ridge, Lasso,
                                   ElasticNet, LogisticRegression)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                                AdaBoostClassifier, GradientBoostingClassifier)
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              accuracy_score, precision_score, recall_score,
                              f1_score, cohen_kappa_score, log_loss,
                              confusion_matrix, classification_report)
from sklearn.datasets import load_breast_cancer
import warnings
warnings.filterwarnings('ignore')

# --- PERSONALIZED SEED ---
first_name = "Dean"
last_name = "DiCarlo"
SEED = sum(ord(c) for c in first_name + last_name)
print(f"Your SEED: {SEED}")
np.random.seed(SEED)

# --- Load Datasets ---
penguins = sns.load_dataset('penguins').dropna().reset_index(drop=True)
tips = sns.load_dataset('tips')
from sklearn.datasets import fetch_california_housing
cal = fetch_california_housing(as_frame=True)
cal_df = cal.frame

print(f"Penguins: {penguins.shape}")
print(f"Tips: {tips.shape}")
print(f"California Housing: {cal_df.shape}")
print("\nSetup complete.")

Your SEED: 1046
Penguins: (333, 7)
Tips: (244, 7)
California Housing: (20640, 9)

Setup complete.


---
# Section 1: Linear Regression (Week 6)
---

## Q1.1 — Simple Linear Regression Fundamentals (12 pts)

Using the California Housing dataset, build a **simple linear regression** predicting `MedHouseVal` from `MedInc` (median income) only.

**A. (4 pts)** Split the data 80/20 using your SEED. Fit a `LinearRegression` model. Print the intercept and slope, rounded to 4 decimal places.

**B. (4 pts)** Compute MAE, RMSE, and R² on the **test set**. Round to 4 decimal places.

**C. (4 pts)** In your markdown answer: interpret the slope in plain English using the actual units (MedInc is in $10,000s, MedHouseVal is in $100,000s). Then explain what the R² value tells you about this model's usefulness.

In [5]:
# YOUR CODE HERE — Q1.1A: Split, fit simple LR, print intercept and slope
linRegModel = LinearRegression()
X = cal_df[["MedInc"]]
y = cal_df[["MedHouseVal"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
linRegModel.fit(X_train, y_train)
print(linRegModel.intercept_)

y_pred = linRegModel.predict(X_test)

MAE = mean_absolute_error(y_test, y_pred)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R2 = r2_score(y_test, y_pred)

[0.45823106]


**YOUR ANSWER Q1.1A:** *(interpret the intercept and slope here)*

In [14]:
# YOUR CODE HERE — Q1.1B: Compute MAE, RMSE, R² on test set
print(MAE)
print(RMSE)
print(R2)
slope = linRegModel.coef_[0]
print(slope)

0.6216252791305877
0.8277581763897586
0.4873460159074715
[0.41536117]


**YOUR ANSWER Q1.1C:** *(interpret slope in real-world units AND explain R²)*

## Q1.2 — Multiple Linear Regression & Weight Interpretation (10 pts)

**A. (5 pts)** Now fit a **multiple linear regression** using ALL 8 features in the California Housing dataset. Use the same train/test split from Q1.1. Print all coefficients with their feature names, sorted by absolute value (largest first). Also print MAE, RMSE, and R².

**B. (5 pts)** In your markdown answer:
1. Which feature has the strongest positive effect? The strongest negative effect?
2. Why might `AveBedrms` have a **negative** coefficient even though more bedrooms usually means a more expensive house?
3. Did adding more features improve R² compared to Q1.1? By how much?

In [ ]:
# YOUR CODE HERE — Q1.2A: Multiple LR, sorted coefficients, metrics


**YOUR ANSWER Q1.2B:** *(answer all 3 interpretation questions)*

## Q1.3 — Loss Functions: Squared vs Absolute vs Huber (8 pts)

**Conceptual — no code needed for parts A-C.**

Consider a model that predicts housing values. Most residuals are small (around ±0.2), but one outlier house has a residual of e = 25.

**A. (2 pts)** Under squared loss, how much more gradient pull does the outlier exert compared to a typical point with e = 1? Show the ratio.

**B. (2 pts)** Under absolute loss, how much more gradient pull does the outlier exert compared to e = 1? Show the ratio.

**C. (2 pts)** Explain what the Huber loss does differently, and what the δ parameter controls.

**D. (2 pts)** Code: Fit a `HuberRegressor` on the California Housing training data from Q1.1 (just `MedInc` as feature). Print its R² on the test set. Is it higher or lower than the OLS R² from Q1.1B? Why might that be?

**YOUR ANSWER Q1.3A-C:** *(loss function analysis)*

In [ ]:
# YOUR CODE HERE — Q1.3D: HuberRegressor
from sklearn.linear_model import HuberRegressor



**YOUR ANSWER Q1.3D:** *(compare Huber R² to OLS R² and explain)*

## Q1.4 — Regularization: Ridge, LASSO, Elastic Net (15 pts)

Using the California Housing data with ALL 8 features. Use the same train/test split.

**A. (3 pts)** Why is standardization **required** before applying regularization? What goes wrong if you skip it?

**B. (6 pts)** Standardize your features (fit on train only!). Fit Ridge, LASSO, and ElasticNet (l1_ratio=0.5), each with α=1.0. For each model, print the coefficients. Identify which coefficients LASSO set to exactly 0 (if any).

**C. (3 pts)** Now fit LASSO with α=0.1 and α=10.0. Print the number of non-zero coefficients for each. Explain the pattern you observe as α increases.

**D. (3 pts)** In your markdown: explain the geometric reason WHY L1 (LASSO) can produce exact zeros but L2 (Ridge) cannot. Use the diamond vs circle analogy.

**YOUR ANSWER Q1.4A:** *(why standardize before regularization?)*

In [ ]:
# YOUR CODE HERE — Q1.4B: Standardize, fit Ridge/LASSO/EN, show coefficients


In [ ]:
# YOUR CODE HERE — Q1.4C: LASSO at different alpha values


**YOUR ANSWER Q1.4C-D:** *(alpha pattern + geometric explanation of L1 vs L2)*

## Q1.5 — kNN Regression & Bias-Variance (8 pts)

**A. (4 pts)** Using the California Housing data (all 8 features, standardized), fit KNeighborsRegressor for k = 1, 5, 15, 50, and 200. For each k, record **train RMSE** and **test RMSE**.

**B. (4 pts)** In your markdown:
1. At k=1, what is the train RMSE? Why is it that value?
2. As k increases, what happens to train RMSE vs test RMSE? Relate this to the bias-variance tradeoff.
3. What value of k would you pick and why?

In [ ]:
# YOUR CODE HERE — Q1.5A: kNN regression for multiple k values


**YOUR ANSWER Q1.5B:** *(bias-variance analysis across k values)*

## Q1.6 — Residual Analysis & Assumptions (6 pts)

**A. (3 pts)** Using your multiple LR model from Q1.2, create TWO plots side by side:
1. Residuals vs Fitted values (scatter plot with horizontal line at 0)
2. Histogram of residuals with a KDE overlay

**B. (3 pts)** In your markdown, evaluate each of the 4 linear regression assumptions based on your plots:
1. Linearity
2. Independence (can you assess this from these plots?)
3. Normality
4. Homoscedasticity (constant variance)

In [ ]:
# YOUR CODE HERE — Q1.6A: Residual plots


**YOUR ANSWER Q1.6B:** *(evaluate all 4 assumptions)*

---
# Section 2: Classification Models & Metrics (Week 7)
---

## Q2.1 — Logistic Regression: From Scratch to sklearn (12 pts)

**A. (3 pts) Conceptual:** Write out the 3-step derivation of the sigmoid function:
1. What transformation takes probability p ∈ (0,1) to odds ∈ (0,∞)?
2. What transformation takes odds to log-odds ∈ (-∞, +∞)?
3. If we set log-odds = w₀ + w₁x and solve for p, what do we get?

**B. (4 pts)** Using the Penguins dataset, create a binary classification: **Adelie vs Chinstrap** only. Use `bill_length_mm` and `bill_depth_mm` as features. Split 80/20 with SEED. Standardize. Fit LogisticRegression. Print the coefficients.

**C. (2 pts)** For a penguin with bill_length_mm = 45 and bill_depth_mm = 18 (after standardization), compute the **predicted probability** of being Chinstrap using `predict_proba`.

**D. (3 pts)** Explain: if you lower the classification threshold from 0.5 to 0.3, what happens to precision and recall? Give a real-world scenario where you'd want to do this.

**YOUR ANSWER Q2.1A:** *(sigmoid derivation steps)*

In [ ]:
# YOUR CODE HERE — Q2.1B: Binary LogReg on penguins


In [ ]:
# YOUR CODE HERE — Q2.1C: Predict probability for specific penguin


**YOUR ANSWER Q2.1D:** *(threshold, precision/recall tradeoff, scenario)*

## Q2.2 — kNN Classification Deep Dive (10 pts)

Using the same Adelie vs Chinstrap subset from Q2.1:

**A. (4 pts)** Fit KNeighborsClassifier with k=1, 3, 7, 15 using **Euclidean** distance and **uniform** weights. For each k, report train accuracy and test accuracy.

**B. (3 pts)** Now fit k=5 with THREE different distance metrics: Manhattan (p=1), Euclidean (p=2), and Chebyshev (p=∞). Report test accuracy for each. Which performed best?

**C. (3 pts)** Conceptual: Draw (describe in words) what the neighborhood shape looks like for Manhattan, Euclidean, and Chebyshev distances in 2D. Why might Manhattan distance work better in high-dimensional data?

In [ ]:
# YOUR CODE HERE — Q2.2A: kNN with varying k


In [ ]:
# YOUR CODE HERE — Q2.2B: kNN with different distance metrics


**YOUR ANSWER Q2.2C:** *(neighborhood shapes + high-dimensional reasoning)*

## Q2.3 — Gaussian Naive Bayes & Priors (10 pts)

Using the **full Penguins dataset** (all 3 species), with `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g` as features.

**A. (4 pts)** Fit a GaussianNB with default priors. Print the learned priors. Then fit another GaussianNB with equal priors [1/3, 1/3, 1/3]. Compare test accuracy of both.

**B. (3 pts) Conceptual:** Write Bayes' rule. Label each term (prior, likelihood, evidence, posterior). Why can we ignore the denominator P(x) when classifying?

**C. (3 pts)** The "naive" assumption says features are conditionally independent given the class. Using the penguins data, compute the correlation between `bill_length_mm` and `flipper_length_mm` within the Adelie class. Is the independence assumption satisfied? Despite this, does GaussianNB still achieve reasonable accuracy?

In [ ]:
# YOUR CODE HERE — Q2.3A: GaussianNB with default vs equal priors


**YOUR ANSWER Q2.3B:** *(Bayes' rule with labels, why ignore denominator)*

In [ ]:
# YOUR CODE HERE — Q2.3C: Within-class correlation check


**YOUR ANSWER Q2.3C:** *(is independence satisfied? does NB still work?)*

## Q2.4 — Loss Functions: Log Loss & Cross-Entropy (8 pts)

**A. (3 pts) Computation by hand:** A binary classifier gives these predictions for 4 test instances:

| Instance | True Label (y) | Predicted P(y=1) |
|----------|---------------|-----------------|
| 1 | 1 | 0.95 |
| 2 | 0 | 0.10 |
| 3 | 1 | 0.60 |
| 4 | 1 | 0.02 |

Compute the **log loss** for each instance separately, then the average. Which instance contributes the most loss? Why?

**B. (3 pts)** Now verify your hand computation with sklearn's `log_loss` function.

**C. (2 pts)** Explain in 2 sentences why cross-entropy is preferred over absolute loss for training classifiers.

**YOUR ANSWER Q2.4A:** *(hand computation of log loss)*

In [ ]:
# YOUR CODE HERE — Q2.4B: Verify log loss with sklearn


**YOUR ANSWER Q2.4C:** *(why cross-entropy over absolute loss?)*

## Q2.5 — Confusion Matrix & Full Metrics Suite (15 pts)

**This is a high-priority exam topic.** Using the full 3-class Penguins dataset:

**A. (4 pts)** Fit a LogisticRegression (max_iter=5000). Generate and display the confusion matrix. Use `classification_report` to show per-class precision, recall, F1.

**B. (3 pts)** From the confusion matrix, manually compute (show your work in markdown):
- Overall accuracy
- Precision for class "Chinstrap"
- Recall for class "Chinstrap"

**C. (4 pts)** Compute Cohen's Kappa by hand:
1. What is observed accuracy?
2. What is expected accuracy? (Show the full computation)
3. What is κ?
4. Verify with `cohen_kappa_score`

**D. (4 pts)** Create an **imbalanced scenario**: take only 5 Chinstrap penguins in training but keep all Adelie and Gentoo. Train a LogReg. What happens to accuracy vs F1-macro? Explain why accuracy is misleading here.

In [ ]:
# YOUR CODE HERE — Q2.5A: Confusion matrix and classification report


**YOUR ANSWER Q2.5B:** *(manual computation of accuracy, precision, recall from CM)*

**YOUR ANSWER Q2.5C:** *(Cohen's kappa by hand)*

In [ ]:
# YOUR CODE HERE — Q2.5C verification + Q2.5D imbalanced scenario


**YOUR ANSWER Q2.5D:** *(why accuracy misleads on imbalanced data)*

---
# Section 3: Support Vector Machines (Week 8)
---

## Q3.1 — Margins: Functional vs Geometric (10 pts)

**A. (5 pts) Conceptual:** A linear SVM has learned weights w = [3, 4] and intercept w₀ = -10. For a test point x = [2, 3] with true label y = +1:

1. Compute the functional margin ŷ·f(x)
2. Compute ||w||
3. Compute the geometric margin
4. Is this point correctly classified?
5. What is the full margin band width (2/||w||)?

**B. (2 pts)** If you multiply all weights by 10 (w = [30, 40], w₀ = -100), recompute the functional margin and geometric margin. Which one changed? Which stayed the same? Why?

**C. (3 pts)** Code: Fit a linear SVC on the Adelie vs Chinstrap penguins (same setup as Q2.1). Print the number of support vectors per class. Use `decision_function` and `np.linalg.norm(clf.coef_)` to compute the margin width.

**YOUR ANSWER Q3.1A:** *(all 5 margin computations)*

**YOUR ANSWER Q3.1B:** *(scaling effect on margins)*

In [ ]:
# YOUR CODE HERE — Q3.1C: Linear SVC, support vectors, margin width


## Q3.2 — Soft Margin & the C Parameter (10 pts)

**A. (4 pts) Conceptual:** For a soft-margin SVM, explain what each slack variable value means:
- ξᵢ = 0
- ξᵢ = 0.4
- ξᵢ = 1.0
- ξᵢ = 2.5

**B. (3 pts)** Using the Breast Cancer dataset, fit SVC(kernel='linear') with C = 0.001, 0.1, 1, 10, 1000. For each C, record: train accuracy, test accuracy, and number of support vectors. Display as a table.

**C. (3 pts)** In your markdown: How does C relate to α in Ridge regularization? As C increases, what happens to the margin width and why? Which C value from your table gives the best bias-variance balance?

**YOUR ANSWER Q3.2A:** *(slack variable interpretations)*

In [ ]:
# YOUR CODE HERE — Q3.2B: SVC with varying C on Breast Cancer
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target



**YOUR ANSWER Q3.2C:** *(C vs α, margin behavior, best C)*

## Q3.3 — Kernels & the Kernel Trick (12 pts)

**A. (3 pts) Conceptual:** Explain the kernel trick in 3 sentences:
1. What problem does it solve?
2. How does the dual formulation make it possible?
3. Give one concrete efficiency example (e.g., degree-2 polynomial on 50 features).

**B. (4 pts)** Generate the `moons` dataset (n_samples=300, noise=0.2, random_state=SEED). Fit SVC with 4 different kernels: linear, poly (degree=3), rbf, and sigmoid. Report test accuracy for each. Which kernel performs best?

**C. (5 pts)** Using the moons dataset with RBF kernel: fit SVC with gamma values [0.01, 0.1, 1, 10, 100] and C=1. For each gamma, record train and test accuracy. Identify which gamma values cause underfitting vs overfitting. What is the "sweet spot"?

**YOUR ANSWER Q3.3A:** *(kernel trick explanation)*

In [ ]:
# YOUR CODE HERE — Q3.3B: Four kernels on moons dataset
from sklearn.datasets import make_moons



In [ ]:
# YOUR CODE HERE — Q3.3C: Gamma exploration with RBF


**YOUR ANSWER Q3.3C:** *(underfitting vs overfitting gamma values, sweet spot)*

## Q3.4 — Support Vector Regression (8 pts)

**A. (3 pts) Conceptual:** Explain the ε-tube in SVR:
1. What is the ε-insensitive loss function? Write it.
2. What happens to a point that falls INSIDE the tube?
3. What happens if you increase ε?

**B. (3 pts)** Using California Housing (just `MedInc` feature), fit SVR with RBF kernel, C=10, epsilon=0.1. Print test RMSE. Then change epsilon to 1.0 and refit. How does RMSE change and why?

**C. (2 pts)** In SVR, minimizing ||w||² means seeking a "flatter" function. Why? (Hint: what IS the gradient of the prediction function?)

**YOUR ANSWER Q3.4A:** *(ε-tube explanation)*

In [ ]:
# YOUR CODE HERE — Q3.4B: SVR with different epsilon values


**YOUR ANSWER Q3.4B-C:** *(epsilon effect + why ||w||² = flatness)*

---
# Section 4: Decision Trees & Ensemble Methods (Week 9)
---

## Q4.1 — Gini Impurity: Compute by Hand (10 pts)

**A. (4 pts)** A node contains 40 instances: 25 Class A, 10 Class B, 5 Class C.
1. Compute the Gini impurity of this node. Show all steps.
2. What would the Gini be if the node were pure (all one class)?
3. What is the maximum possible Gini for a 3-class problem?

**B. (6 pts)** This node is split into two children:
- Left child: [20 A, 2 B, 0 C] = 22 instances
- Right child: [5 A, 8 B, 5 C] = 18 instances

1. Compute Gini for the left child
2. Compute Gini for the right child
3. Compute the **weighted Gini** of this split
4. Is this split good or bad? Compare to the parent Gini from part A.

**YOUR ANSWER Q4.1A:** *(Gini computation for parent node)*

**YOUR ANSWER Q4.1B:** *(weighted Gini computation for the split)*

In [ ]:
# YOUR CODE HERE — Verify your hand calculations with code
# Parent node
p_A, p_B, p_C = 25/40, 10/40, 5/40
gini_parent = 1 - (p_A**2 + p_B**2 + p_C**2)
print(f"Parent Gini: {gini_parent:.4f}")

# Left child
# ... continue verification


## Q4.2 — Decision Trees: Overfitting & Pruning (12 pts)

Using the Breast Cancer dataset:

**A. (4 pts)** Fit a DecisionTreeClassifier with NO constraints (default parameters). Report train and test accuracy. Is this tree overfitting? How can you tell from the numbers?

**B. (4 pts)** Now fit trees with `max_depth` = 1, 2, 3, 5, 10, None. For each, record train and test accuracy. Plot both curves on the same graph (x-axis = max_depth). Identify the sweet spot.

**C. (4 pts)** Use `cost_complexity_pruning_path` to find the best `ccp_alpha`. Fit a pruned tree. Compare its test accuracy to the unpruned tree and the best max_depth tree. Write out the pruning formula R_α(T) = R(T) + α|T̃| and explain each term.

In [ ]:
# YOUR CODE HERE — Q4.2A: Unconstrained decision tree


**YOUR ANSWER Q4.2A:** *(is it overfitting? how do you know?)*

In [ ]:
# YOUR CODE HERE — Q4.2B: Trees with varying max_depth + plot


In [ ]:
# YOUR CODE HERE — Q4.2C: Cost-complexity pruning


**YOUR ANSWER Q4.2C:** *(pruning formula explanation + comparison)*

## Q4.3 — Regression Trees: Piecewise-Constant Output (6 pts)

**A. (3 pts)** Fit a DecisionTreeRegressor (max_depth=3) on California Housing using only `MedInc`. Create a scatter plot of actual vs predicted values. Describe the pattern you see — why does it look like horizontal bands?

**B. (3 pts)** How does a regression tree's output fundamentally differ from linear regression's output? Why can't a regression tree predict a value it has never seen in the training set's target?

In [ ]:
# YOUR CODE HERE — Q4.3A: Regression tree + actual vs predicted plot


**YOUR ANSWER Q4.3B:** *(piecewise-constant vs continuous, why limited to training means)*

## Q4.4 — Bootstrap Sampling & Bagging (10 pts)

**A. (3 pts) Conceptual:**
1. In bootstrap sampling, what fraction of data is left out on average? Derive using the formula P(not selected) = (1 - 1/n)^n.
2. What is this leftover data called, and why is it useful?

**B. (4 pts)** Using Breast Cancer data, fit a BaggingClassifier with n_estimators=100 and `oob_score=True`. Print the OOB score and test accuracy. How do they compare?

**C. (3 pts)** Fit BaggingClassifier with n_estimators = 1, 5, 10, 50, 100, 500. Plot test accuracy vs number of trees. Does accuracy keep improving? Does it ever get worse (overfit)? Why or why not?

**YOUR ANSWER Q4.4A:** *(bootstrap derivation + OOB explanation)*

In [ ]:
# YOUR CODE HERE — Q4.4B: Bagging with OOB score


In [ ]:
# YOUR CODE HERE — Q4.4C: Bagging with varying n_estimators


**YOUR ANSWER Q4.4C:** *(does bagging overfit with more trees?)*

## Q4.5 — Random Forest vs Bagging (10 pts)

**A. (5 pts)** Using Breast Cancer data (30 features), fit BOTH:
1. BaggingClassifier(n_estimators=100)
2. RandomForestClassifier(n_estimators=100)

Use the SAME random_state=SEED. Report test accuracy for both. Then get feature importances from the RF model — print the top 5 features by importance.

**B. (2 pts)** The ONLY difference between Bagging and RF is ___. Fill in the blank and explain WHY this difference helps (use the word "decorrelate").

**C. (3 pts)** Conceptual: If your dataset has one very dominant feature (like `flipper_length_mm` for penguins), would you expect Bagging or RF to benefit more? Why?

In [ ]:
# YOUR CODE HERE — Q4.5A: Bagging vs RF comparison + feature importances


**YOUR ANSWER Q4.5B-C:** *(the ONE difference + decorrelation + dominant feature scenario)*

## Q4.6 — Boosting: AdaBoost & Gradient Boosting (15 pts)

**A. (4 pts) Conceptual — AdaBoost:** Given a stump with weighted error ε = 0.3:
1. Compute αₘ = ln((1-ε)/ε). Show your work.
2. What happens to αₘ when ε = 0.5 (random guessing)?
3. What happens when ε ≈ 0? What does this stump become?
4. What does AdaBoost do to the weights of misclassified samples after this stump?

**B. (4 pts)** Conceptual — Gradient Boosting: 
1. What does each new tree in gradient boosting fit? (Not the original target!)
2. Write the update formula: T_m = ?
3. What does ν (learning rate) control?
4. Why is it called "gradient descent in function space"?

**C. (4 pts)** Using Breast Cancer data, fit AdaBoostClassifier and GradientBoostingClassifier (both with n_estimators=100). Compare test accuracies. Then fit GradientBoosting with learning_rate = 0.01, 0.1, 0.5, 1.0 and report test accuracy for each.

**D. (3 pts)** Fill in this table (True/False + brief justification):

| Statement | T/F | Why? |
|-----------|-----|------|
| Bagging can overfit with 10,000 trees | | |
| Gradient Boosting can overfit with 10,000 trees | | |
| AdaBoost trees are trained independently | | |
| Random Forest trees are trained independently | | |

**YOUR ANSWER Q4.6A:** *(AdaBoost α computation + extreme cases)*

**YOUR ANSWER Q4.6B:** *(Gradient Boosting conceptual answers)*

In [ ]:
# YOUR CODE HERE — Q4.6C: AdaBoost vs GradientBoosting + learning rate sweep


**YOUR ANSWER Q4.6D:** *(True/False table with justifications)*

---
# Section 5: Model Validation & Cross-Validation (Week 10)
---

## Q5.1 — Why Single Splits Are Unreliable (6 pts)

**A. (3 pts)** Using the Breast Cancer dataset, fit a LogisticRegression with 5 DIFFERENT random seeds for the train/test split (seeds: SEED, SEED+1, SEED+2, SEED+3, SEED+4). Report the test accuracy for each. Compute the range (max - min).

**B. (3 pts)** In your markdown: Why is reporting any single one of these numbers misleading? How does cross-validation solve this problem?

In [ ]:
# YOUR CODE HERE — Q5.1A: Same model, 5 different splits


**YOUR ANSWER Q5.1B:** *(why single splits are unreliable, how CV helps)*

## Q5.2 — k-Fold Cross-Validation Mechanics (12 pts)

**A. (3 pts) Computation:** You have a dataset with 2,000 instances and use 5-fold CV.
1. How many instances are in each fold?
2. How many instances are used for training in each iteration?
3. How many total models are fitted?

**B. (4 pts)** Using Breast Cancer data, perform 10-fold Stratified CV on a LogisticRegression. Print each fold's score, the mean, and the std. Use `cross_validate` with `return_train_score=True`.

**C. (2 pts)** Why is it critical to use **StratifiedKFold** instead of regular KFold for classification? Give a concrete example of what could go wrong.

**D. (3 pts)** You want to compare LogReg, kNN(k=5), and GaussianNB on the same data. Write code that uses the **same StratifiedKFold object** for all three. Why is this important for a fair comparison?

**YOUR ANSWER Q5.2A:** *(fold mechanics computation)*

In [ ]:
# YOUR CODE HERE — Q5.2B: 10-fold Stratified CV


**YOUR ANSWER Q5.2C:** *(why stratified?)*

In [ ]:
# YOUR CODE HERE — Q5.2D: Same folds for 3 models


**YOUR ANSWER Q5.2D:** *(why same folds matter)*

## Q5.3 — LOOCV (6 pts)

**A. (3 pts) Conceptual:**
1. In LOOCV with n=500, how many models are fitted?
2. What is the training set size for each iteration?
3. Name two situations where LOOCV is appropriate despite its cost.

**B. (3 pts)** Compare: 10-fold CV on a dataset of 10,000 instances vs LOOCV on the same data. How many models does each fit? Which is more practical and why?

**YOUR ANSWER Q5.3:** *(LOOCV mechanics and comparison)*

## Q5.4 — GridSearchCV & Model Selection Workflow (15 pts)

This is the capstone question. It tests the **complete workflow**.

**A. (2 pts)** Using Breast Cancer data, split off a 20% test set (locked vault). This test set will NOT be touched until the very end.

**B. (5 pts)** Define a StratifiedKFold(n_splits=5). Use GridSearchCV to tune:
- KNeighborsClassifier: n_neighbors = [1, 3, 5, 7, 9, 11, 15]
- SVC (RBF): C = [0.1, 1, 10], gamma = [0.01, 0.1, 1]

Print `best_params_` and `best_score_` for each.

**C. (2 pts)** How many total models were fitted for kNN? For SVC? Show the math.

**D. (3 pts)** Compare the best kNN vs best SVC using the SAME CV folds. Which has higher mean CV score? Which has lower standard deviation? Which would you select and why?

**E. (3 pts)** Take your selected model, evaluate ONCE on the test set. Report the final accuracy. Explain why this number may differ from the CV score, and why you can only do this step ONCE.

In [ ]:
# YOUR CODE HERE — Q5.4A: Split off test set


In [ ]:
# YOUR CODE HERE — Q5.4B: GridSearchCV for kNN and SVC


**YOUR ANSWER Q5.4C:** *(model count math)*

In [ ]:
# YOUR CODE HERE — Q5.4D: Compare best models on same folds


In [ ]:
# YOUR CODE HERE — Q5.4E: Final test evaluation


**YOUR ANSWER Q5.4E:** *(why the final test number matters and why only once)*

## Q5.5 — Validation Curves & One-Standard-Error Rule (8 pts)

**A. (4 pts)** Using the Breast Cancer data training set, plot a validation curve for KNeighborsClassifier varying `n_neighbors` from 1 to 29 (odd values only). Plot both train and validation scores with error bands (mean ± std). Label the regions of overfitting, good fit, and underfitting.

**B. (2 pts)** Apply the one-standard-error rule: which k would you select? Explain the rule.

**C. (2 pts)** Conceptual: For kNN, which direction is "simpler" — small k or large k? For a decision tree's max_depth, which direction is "simpler"? Why do these go in opposite directions?

In [ ]:
# YOUR CODE HERE — Q5.5A: Validation curve plot


**YOUR ANSWER Q5.5B:** *(one-standard-error rule application)*

**YOUR ANSWER Q5.5C:** *(simplicity direction for kNN vs trees)*

---
# Section 6: Cross-Topic Connections & Tricky Concepts
---

These questions test your ability to connect ideas across weeks — Professor Sahami loves these.

## Q6.1 — The Regularization Family (6 pts)

Fill in this table connecting regularization concepts across models:

| Model | What controls complexity? | More complex (overfit direction) | Simpler (underfit direction) |
|-------|--------------------------|--------------------------------|------------------------------|
| Ridge | α | α → ? | α → ? |
| LASSO | α | α → ? | α → ? |
| SVM | C | C → ? | C → ? |
| kNN | k | k → ? | k → ? |
| Decision Tree | max_depth | max_depth → ? | max_depth → ? |
| Random Forest | n_estimators | Adding trees → ? | Removing trees → ? |

**Key connection:** How does C in SVM relate to α in Ridge? (This appeared on Midterm 1.)

**YOUR ANSWER Q6.1:** *(completed table + C vs α connection)*

## Q6.2 — Data Leakage Traps (6 pts)

For each scenario, identify whether there is data leakage and explain why:

1. You standardize ALL your data, then split into train/test, then fit your model.
2. You split into train/test, fit a scaler on training data, transform both train and test with it.
3. You use GridSearchCV on your training set, then evaluate the best model on your test set once.
4. You run GridSearchCV, don't like the result, tweak the param_grid, run again, evaluate on test set, repeat until test accuracy is good.
5. You use LOOCV and the scaler is fit inside each fold (only on the n-1 training points).

For each scenario, state: **LEAK** or **NO LEAK**, and explain in one sentence.

**YOUR ANSWER Q6.2:** *(5 leakage assessments)*

## Q6.3 — Model Properties Comparison (8 pts)

Answer TRUE or FALSE and provide a one-sentence justification:

1. Decision trees require feature standardization.
2. kNN stores the entire training set and uses it at prediction time.
3. Logistic regression can only produce linear decision boundaries.
4. Random Forest's feature importance (MDI) measures the same thing as LASSO's coefficient magnitude.
5. SVMs with RBF kernel map data to an infinite-dimensional feature space.
6. Naive Bayes assumes features are independent of EACH OTHER (unconditionally).
7. Gradient Boosting trees are trained independently of each other.
8. The test set should be used to select the best hyperparameters.

**YOUR ANSWER Q6.3:** *(8 True/False with justifications)*

## Q6.4 — Debugging a Pipeline (8 pts)

The following pipeline has **4 deliberate mistakes**. Find them, explain why each is wrong, and write the corrected code.

```python
# BAD CODE — FIND THE BUGS
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

X = penguins[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']]
y = penguins['species']

# Step 1: Scale everything first
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 2: Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

# Step 3: Fit model
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X_train, y_train)

# Step 4: Evaluate
print(f"Train accuracy: {knn.score(X_train, y_train):.4f}")
print(f"This is our final reported performance.")
```

**YOUR ANSWER Q6.4:** *(identify all 4 bugs, explain each, write corrected code)*

In [ ]:
# YOUR CODE HERE — Write the corrected pipeline


---
# Self-Assessment Checklist

Before you consider yourself ready for the exam, make sure you can:

- [ ] Compute Gini impurity and weighted Gini by hand
- [ ] Compute log loss by hand for individual instances
- [ ] Compute Cohen's kappa by hand (observed and expected accuracy)
- [ ] Compute functional and geometric margins by hand
- [ ] Explain the sigmoid derivation from log-odds
- [ ] Write the Ridge, LASSO, and Elastic Net loss formulas from memory
- [ ] Explain why LASSO zeros coefficients but Ridge doesn't (geometry)
- [ ] Describe what each slack variable value (ξ) means
- [ ] Explain the kernel trick in ≤3 sentences
- [ ] Distinguish bagging vs RF vs AdaBoost vs gradient boosting
- [ ] Run a complete ML pipeline: split → scale (fit on train!) → fit → evaluate
- [ ] Use GridSearchCV and compute total models fitted
- [ ] Read a validation curve and identify overfitting/underfitting
- [ ] Know the one-standard-error rule
- [ ] Identify data leakage scenarios
- [ ] Connect C (SVM) to α (regularization)

**Good luck, Dean. You've got this.**
---